In [1]:
import pandas as pd

# País LATAM | Nombre | Ciudad | Estado / Provincia / Departamento
locations = pd.read_excel('ArchivosExcel/Lugares.xlsx')

print("Columnas encontradas:", locations.columns.tolist())

if 'Nombre' not in locations.columns:
    raise ValueError("El archivo debe contener una columna llamada 'Nombre'.")

locations.head()


Columnas encontradas: ['País LATAM', 'Nombre', 'Ciudad', 'Estado / Provincia / Departamento']


,País LATAM,Nombre,Ciudad,Estado / Provincia / Departamento
0,MX,IMSS Hospital Centro Medico Siglo XXI,NaN,NaN
1,MX,Ángeles Metropolitano,Ciudad de México,NaN
2,MX,Hospital de alta especialidad del Bajío,NaN,Guanajuato


In [2]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from urllib.parse import quote_plus
import re
import time

HEADLESS = False

options = webdriver.ChromeOptions()
if HEADLESS:
    options.add_argument('--headless=new')

options.add_argument('--window-size=1920,1080')
options.add_argument('--lang=es-MX')

driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 15)


In [3]:
def clean_value(value):
    if pd.isna(value):
        return ''
    return str(value).strip()


def build_search_query(row):
    parts = [
        clean_value(row.get('Nombre', '')),
        clean_value(row.get('Ciudad', '')),
        clean_value(row.get('Estado / Provincia / Departamento', '')),
        clean_value(row.get('País LATAM', ''))
    ]
    return ' '.join(part for part in parts if part)


def detect_captcha(driver):
    try:
        text = driver.find_element(By.TAG_NAME, 'body').text.lower()
    except Exception:
        return False

    indicators = [
        'unusual traffic',
        'tráfico inusual',
        'captcha',
        'nuestros sistemas han detectado'
    ]
    return any(item in text for item in indicators)


def dismiss_consent_if_present(driver):
    # No intenta saltarse un CAPTCHA. Solo cierra, si aparece, el diálogo normal de consentimiento.
    xpaths = [
        "//button//*[contains(text(), 'Aceptar todo')]/ancestor::button",
        "//button[contains(., 'Aceptar todo')]",
        "//button//*[contains(text(), 'Accept all')]/ancestor::button",
        "//button[contains(., 'Accept all')]"
    ]

    for xpath in xpaths:
        try:
            buttons = driver.find_elements(By.XPATH, xpath)
            if buttons:
                buttons[0].click()
                time.sleep(1)
                return True
        except Exception:
            pass
    return False


def detail_panel_is_open(driver):
    selectors = [
        'h1.DUwDvf',
        'h1[class*="fontHeadlineLarge"]'
    ]

    for selector in selectors:
        try:
            elements = driver.find_elements(By.CSS_SELECTOR, selector)
            if any(e.is_displayed() and e.text.strip() for e in elements):
                return True
        except Exception:
            pass
    return False


def open_first_result_if_needed(driver, timeout=12):
    # Si Maps abrió directamente una ficha, no hacemos nada
    if detail_panel_is_open(driver):
        return True

    end_time = time.time() + timeout

    while time.time() < end_time:
        if detail_panel_is_open(driver):
            return True

        result_selectors = [
            'a.hfpxzc',
            'div[role="feed"] a[href*="/maps/place/"]',
            'a[href*="/maps/place/"]'
        ]

        for selector in result_selectors:
            try:
                results = [
                    e for e in driver.find_elements(By.CSS_SELECTOR, selector)
                    if e.is_displayed()
                ]

                if results:
                    driver.execute_script(
                        "arguments[0].scrollIntoView({block: 'center'});",
                        results[0]
                    )
                    time.sleep(0.3)
                    results[0].click()

                    try:
                        WebDriverWait(driver, 10).until(
                            lambda d: detail_panel_is_open(d)
                        )
                    except TimeoutException:
                        pass

                    return detail_panel_is_open(driver)
            except Exception:
                pass

        time.sleep(0.5)

    return detail_panel_is_open(driver)


def get_google_name(driver):
    selectors = [
        'h1.DUwDvf',
        'h1[class*="fontHeadlineLarge"]'
    ]

    for selector in selectors:
        try:
            element = WebDriverWait(driver, 5).until(
                EC.visibility_of_element_located((By.CSS_SELECTOR, selector))
            )
            value = element.text.strip()
            if value:
                return value
        except Exception:
            pass

    return 'Name not found'


def get_address_from_maps(driver):
    selectors = [
        'button[data-item-id="address"]',
        '[data-item-id="address"]',
        'button[aria-label^="Dirección:"]',
        'button[aria-label^="Address:"]'
    ]

    for selector in selectors:
        try:
            elements = driver.find_elements(By.CSS_SELECTOR, selector)
            for element in elements:
                if not element.is_displayed():
                    continue

                value = (
                    element.get_attribute('aria-label')
                    or element.text
                    or ''
                ).strip()

                if value:
                    value = re.sub(
                        r'^(Address|Dirección):\s*',
                        '',
                        value,
                        flags=re.IGNORECASE
                    ).strip()

                    if value:
                        return value
        except Exception:
            pass

    return 'Address not found'


def extract_postal_code(address):
    if not address or address == 'Address not found':
        return 'Postal code not found'

    # México y muchos países LATAM usan códigos postales de 4 a 6 dígitos.
    matches = re.findall(r'(?<!\d)(\d{4,6})(?!\d)', address)
    if matches:
        return matches[-1]

    return 'Postal code not found'


def valid_coordinates(latitude, longitude):
    try:
        latitude = float(latitude)
        longitude = float(longitude)
        return -90 <= latitude <= 90 and -180 <= longitude <= 180
    except (TypeError, ValueError):
        return False


def extract_exact_coordinates_from_url(url):
    if not url:
        return None, None

    # En muchas fichas de Maps las coordenadas exactas aparecen como !3dLAT!4dLON
    match = re.search(
        r'!3d(-?\d+(?:\.\d+)?)!4d(-?\d+(?:\.\d+)?)',
        url
    )

    if match:
        lat, lon = float(match.group(1)), float(match.group(2))
        if valid_coordinates(lat, lon):
            return lat, lon

    return None, None


def extract_viewport_center(url):
    if not url:
        return None, None

    match = re.search(
        r'@(-?\d+(?:\.\d+)?),(-?\d+(?:\.\d+)?)',
        url
    )

    if match:
        lat, lon = float(match.group(1)), float(match.group(2))
        if valid_coordinates(lat, lon):
            return lat, lon

    return None, None


def _normalize_page_source(source):
    if not source:
        return ''

    normalized = source
    for _ in range(2):
        normalized = normalized.replace('\\\"', '"')
        normalized = normalized.replace('\\"', '"')

    replacements = {
        r'\u0022': '"',
        r'\u003d': '=',
        r'\u0026': '&',
        '&quot;': '"',
        '&#34;': '"'
    }

    for old, new in replacements.items():
        normalized = normalized.replace(old, new)

    return normalized


def extract_structured_coordinates_from_page(driver):
    try:
        source = _normalize_page_source(driver.page_source)
    except Exception:
        return []

    candidates = []

    patterns = [
        r'"latitude"\s*:\s*(-?\d+(?:\.\d+)?)\s*,\s*"longitude"\s*:\s*(-?\d+(?:\.\d+)?)',
        r'"latitude"\s*:\s*(-?\d+(?:\.\d+)?).{0,180}?"longitude"\s*:\s*(-?\d+(?:\.\d+)?)',
        r'"longitude"\s*:\s*(-?\d+(?:\.\d+)?).{0,180}?"latitude"\s*:\s*(-?\d+(?:\.\d+)?)'
    ]

    for i, pattern in enumerate(patterns):
        for match in re.finditer(pattern, source, flags=re.DOTALL):
            if i < 2:
                lat, lon = float(match.group(1)), float(match.group(2))
            else:
                lon, lat = float(match.group(1)), float(match.group(2))

            if valid_coordinates(lat, lon):
                candidates.append((lat, lon))

    for match in re.finditer(
        r'!3d(-?\d+(?:\.\d+)?)!4d(-?\d+(?:\.\d+)?)',
        source
    ):
        lat, lon = float(match.group(1)), float(match.group(2))
        if valid_coordinates(lat, lon):
            candidates.append((lat, lon))

    unique = []
    seen = set()
    for lat, lon in candidates:
        key = (round(lat, 7), round(lon, 7))
        if key not in seen:
            seen.add(key)
            unique.append((lat, lon))

    return unique


def choose_best_candidate(candidates, driver):
    if not candidates:
        return None, None

    if len(candidates) == 1:
        return candidates[0]

    # Una vez que YA confirmamos que la ficha está abierta, el viewport sirve
    # solamente para escoger entre candidatos encontrados en el HTML.
    center_lat, center_lon = extract_viewport_center(driver.current_url)

    if center_lat is None or center_lon is None:
        return candidates[0]

    def squared_distance(candidate):
        lat, lon = candidate
        return (lat - center_lat) ** 2 + (lon - center_lon) ** 2

    return min(candidates, key=squared_distance)


def get_coordinates_from_maps(driver):
    # IMPORTANTE: esta función solo debe llamarse después de confirmar
    # que hay una ficha de lugar abierta.
    lat, lon = extract_exact_coordinates_from_url(driver.current_url)
    if lat is not None:
        return lat, lon, 'CURRENT_URL_EXACT'

    end_time = time.time() + 5
    page_candidates = []

    while time.time() < end_time:
        page_candidates = extract_structured_coordinates_from_page(driver)
        if page_candidates:
            break
        time.sleep(0.5)

    if page_candidates:
        lat, lon = choose_best_candidate(page_candidates, driver)
        return lat, lon, 'PAGE_SOURCE'

    # Último recurso. Lo dejamos marcado como REVIEW porque @LAT,LON
    # representa el centro visible del mapa y no siempre el punto exacto.
    lat, lon = extract_viewport_center(driver.current_url)
    if lat is not None:
        return lat, lon, 'VIEWPORT_REVIEW'

    return None, None, 'NOT_FOUND'


def search_google_maps(row):
    query = build_search_query(row)

    empty_result = {
        'Google_Name': '',
        'Google_Address': '',
        'Postal_Code': '',
        'Latitude': None,
        'Longitude': None,
        'Coordinate_Source': 'NOT_FOUND',
        'Search_Query': query,
        'Search_Status': 'EMPTY INPUT'
    }

    if not query:
        return empty_result

    print(f'Buscando: {query}')

    try:
        # URL de búsqueda de Google Maps. Es mucho más confiable que /maps/place/<texto>.
        url = (
            'https://www.google.com/maps/search/?api=1&query='
            f'{quote_plus(query)}'
        )
        driver.get(url)

        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.TAG_NAME, 'body'))
        )

        dismiss_consent_if_present(driver)
        time.sleep(1)

        if detect_captcha(driver):
            return {
                **empty_result,
                'Search_Status': 'STOP - CAPTCHA / MANUAL REVIEW'
            }

        opened = open_first_result_if_needed(driver, timeout=12)

        if not opened:
            print('  No se pudo abrir una ficha de lugar.')
            return {
                **empty_result,
                'Search_Status': 'NO PLACE DETAIL FOUND'
            }

        # Damos un momento corto para que se llenen dirección y URL final.
        time.sleep(1.2)

        google_name = get_google_name(driver)
        address = get_address_from_maps(driver)
        postal_code = extract_postal_code(address)
        latitude, longitude, coordinate_source = get_coordinates_from_maps(driver)

        if google_name == 'Name not found' and address == 'Address not found':
            status = 'REVIEW - PLACE DETAIL INCOMPLETE'
        elif address == 'Address not found':
            status = 'REVIEW - ADDRESS NOT FOUND'
        elif latitude is None or longitude is None:
            status = 'ADDRESS FOUND - COORDINATES REVIEW'
        elif coordinate_source == 'VIEWPORT_REVIEW':
            status = 'COORDINATES FROM VIEWPORT - REVIEW'
        elif postal_code == 'Postal code not found':
            status = 'ADDRESS FOUND - CP REVIEW'
        else:
            status = 'OK'

        print(f'  Nombre: {google_name}')
        print(f'  Dirección: {address}')
        print(f'  CP: {postal_code}')
        print(f'  Latitud: {latitude}')
        print(f'  Longitud: {longitude}')
        print(f'  Fuente coordenadas: {coordinate_source}')
        print(f'  Estado: {status}')
        print(f'  URL final: {driver.current_url}')

        return {
            'Google_Name': google_name,
            'Google_Address': address,
            'Postal_Code': postal_code,
            'Latitude': latitude,
            'Longitude': longitude,
            'Coordinate_Source': coordinate_source,
            'Search_Query': query,
            'Search_Status': status
        }

    except Exception as e:
        print(f'Error con {query}: {type(e).__name__}: {e}')

        return {
            **empty_result,
            'Google_Address': 'Error',
            'Search_Status': f'ERROR - {type(e).__name__}'
        }


In [4]:
# Cuando la prueba anterior funcione, procesa todas las filas.

all_results = []

for idx, row in locations.iterrows():
    result = search_google_maps(row)
    all_results.append(result)

    # Si Google muestra CAPTCHA, es mejor detenerse y continuar manualmente después.
    if result['Search_Status'].startswith('STOP - CAPTCHA'):
        print(f'Proceso detenido en la fila {idx} por CAPTCHA.')
        break

    # Pausa moderada entre búsquedas.
    time.sleep(1.5)

# Agregar resultados
for i, result in enumerate(all_results):
    locations.loc[locations.index[i], 'Google_Name'] = result['Google_Name']
    locations.loc[locations.index[i], 'Google_Address'] = result['Google_Address']
    locations.loc[locations.index[i], 'Postal_Code'] = result['Postal_Code']
    locations.loc[locations.index[i], 'Latitude'] = result['Latitude']
    locations.loc[locations.index[i], 'Longitude'] = result['Longitude']
    locations.loc[locations.index[i], 'Coordinate_Source'] = result['Coordinate_Source']
    locations.loc[locations.index[i], 'Search_Query'] = result['Search_Query']
    locations.loc[locations.index[i], 'Search_Status'] = result['Search_Status']

locations[
    [
        'Nombre',
        'Google_Name',
        'Google_Address',
        'Postal_Code',
        'Latitude',
        'Longitude',
        'Coordinate_Source',
        'Search_Status'
    ]
].head(20)


Buscando: IMSS Hospital Centro Medico Siglo XXI MX
  Nombre: Centro Médico Nacional Siglo XXI
  Dirección: Av. Cuauhtémoc 330, Doctores, Cuauhtémoc, 06720 Ciudad de México, CDMX
  CP: 06720
  Latitud: 19.4083431
  Longitud: -99.1539944
  Fuente coordenadas: CURRENT_URL_EXACT
  Estado: OK
  URL final: https://www.google.com/maps/place/Centro+M%C3%A9dico+Nacional+Siglo+XXI/@19.4083431,-104.0319241,7z/data=!4m10!1m2!2m1!1sIMSS+Hospital+Centro+Medico+Siglo+XXI+MX!3m6!1s0x85d1ff186212332f:0xb3f49e0900720c7e!8m2!3d19.4083431!4d-99.1539944!15sCihJTVNTIEhvc3BpdGFsIENlbnRybyBNZWRpY28gU2lnbG8gWFhJIE1YWioiKGltc3MgaG9zcGl0YWwgY2VudHJvIG1lZGljbyBzaWdsbyB4eGkgbXiSAQhob3NwaXRhbJoBI0NoWkRTVWhOTUc5blMwVkpRMEZuU1VOYWVIWnVVR1ZuRUFF4AEA-gEECAAQQA!16s%2Fg%2F11bx55_vnn?entry=ttu&g_ep=EgoyMDI2MDkyNy4xIKXMDSoASAFQAw%3D%3D
Buscando: Ángeles Metropolitano Ciudad de México MX
  Nombre: Hospital Angeles Metropolitano
  Dirección: Tlacotalpan 59, Centro Urbano Pdte. Juárez, Roma Sur, Cuauhtémoc, 06760 Cuauhtémoc, 

,Nombre,Google_Name,Google_Address,Postal_Code,Latitude,Longitude,Coordinate_Source,Search_Status
0,IMSS Hospital Centro Medico Siglo XXI,Centro Médico Nacional Siglo XXI,"Av. Cuauhtémoc 330, Doctores, Cuauhtémoc, 0672...",06720,19.408343,-99.153994,CURRENT_URL_EXACT,OK
1,Ángeles Metropolitano,Hospital Angeles Metropolitano,"Tlacotalpan 59, Centro Urbano Pdte. Juárez, Ro...",06760,19.406797,-99.166871,PAGE_SOURCE,OK
2,Hospital de alta especialidad del Bajío,Hospital Regional de Alta Especialidad del Bajío,"Blvd. Milenio 130, San Carlos la Roncha, 37544...",37544,21.061359,-101.576600,PAGE_SOURCE,OK


In [5]:
# Guardar el archivo enriquecido.
locations.to_excel(
    'ArchivosExcel/DireccionesRevisadas.xlsx',
    index=False
)

print('Archivo guardado en: ArchivosExcel/DireccionesRevisadas.xlsx')

# Cerrar Chrome al terminar.
driver.quit()


Archivo guardado en: ArchivosExcel/DireccionesRevisadas.xlsx
